## 1. Load the raw data

In [ ]:
import pandas as pd
import numpy as np

RAW_PATH = "customer_churn_raw_v2.csv"
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()
log = []   

print("Shape:", df.shape)
df.head()

: 

## 2. Inspect: missing values, duplicates, data types

In [ ]:
print(df.dtypes, "\n")
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated CustomerIDs:", df["CustomerID"].duplicated().sum())
df.describe(include="all").T

In [ ]:

print("Non-numeric values in TotalChargesINR:")
print(df.loc[pd.to_numeric(df["TotalChargesINR"], errors="coerce").isna(), "TotalChargesINR"].value_counts(), "\n")


for c in ["Region","SubscriptionType","ContractType","PaymentMethod","AutoPay","Churn"]:
    print(c, df[c].value_counts(dropna=False).to_dict())

### Findings
| # | Issue | Evidence |
|---|-------|----------|
| 1 | Duplicate records | 5 rows are exact copies of earlier rows (same CustomerID) |
| 2 | Missing values | `Region` has 13 nulls |
| 3 | Type mismatch | `TotalChargesINR` is text because 11 rows contain `"unknown"` |
| 4 | Outliers / unit error | 7 `MonthlyChargesINR` values are about 12x too high (see section 6) |
| 5 | Inconsistent headers | CamelCase, with unit suffixes mixed into names |
| 6 | Date columns | None in this dataset, so no date reformatting is needed |

## 3. Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
assert df["CustomerID"].is_unique
log.append(("Duplicates", "Exact duplicate rows (same CustomerID)", "Dropped, kept first occurrence", before - len(df)))
print(f"Removed {before-len(df)} duplicate rows -> {len(df)} remain")

## 4. Standardize column headers and text

In [ ]:
df.columns = (df.columns.str.strip()
                .str.replace(r"(?<=[a-z])(?=[A-Z])", "_", regex=True)
                .str.lower())
df = df.rename(columns={"customer_i_d":"customer_id", "monthly_charges_i_n_r":"monthly_charges_inr",
                        "total_charges_i_n_r":"total_charges_inr"})
print(list(df.columns))

cat_cols = ["region","subscription_type","contract_type","payment_method","auto_pay","churn"]
for c in cat_cols:
    df[c] = df[c].astype("string").str.strip().str.title()
df["customer_id"] = df["customer_id"].str.strip().str.upper()
df["payment_method"] = df["payment_method"].replace({"Netbanking":"Net Banking", "Upi":"UPI"})
df["contract_type"] = df["contract_type"].replace({"Month-To-Month":"Month-to-Month"})
log.append(("Standardization", "CamelCase headers, mixed text formatting", "snake_case headers; trimmed + Title Case categories", len(cat_cols)))

## 5. Fix data types and handle `total_charges_inr`
`"unknown"` is a placeholder for a missing number. It is converted to `NaN`, then imputed.
**Reasoning:** across the valid rows, `total_charges ~ monthly_charges x tenure_months` (median ratio about 0.98), so a customer's own billing history gives a better estimate than a global mean. Tenure 0 means no billing yet, so the total is 0.

In [ ]:
df["total_charges_inr"] = pd.to_numeric(df["total_charges_inr"], errors="coerce")
n_missing_total = df["total_charges_inr"].isna().sum()
print("Missing total_charges_inr:", n_missing_total)

## 6. Outliers in `monthly_charges_inr`
Seven values (Rs 5,164 to Rs 13,858) are far above their plan's typical range (Basic about Rs 450, Standard about Rs 750, Pro about Rs 1,150).
For the six of those rows with a valid total, `total_charges / (monthly x tenure)` is about 0.08, which is roughly 1/12. That points to an **annual amount entered in the monthly column**, not a real price. Correction: divide by 12.

In [ ]:

tier_median = df.groupby("subscription_type")["monthly_charges_inr"].transform("median")
outlier = df["monthly_charges_inr"] > 3 * tier_median

check = (df["total_charges_inr"] / (df["monthly_charges_inr"] * df["tenure_months"]))[outlier]
print("Outlier rows:", outlier.sum(), "| total/(monthly*tenure) ratio on these rows:\n", check.round(3).to_string())

df.loc[outlier, "monthly_charges_inr"] = (df.loc[outlier, "monthly_charges_inr"] / 12).round(2)
log.append(("Outliers", "Monthly charge ~12x tier median (annual value in monthly field)", "Divided by 12", int(outlier.sum())))


miss = df["total_charges_inr"].isna()
df.loc[miss, "total_charges_inr"] = (df.loc[miss, "monthly_charges_inr"] * df.loc[miss, "tenure_months"]).round(2)
log.append(("Missing values", "total_charges_inr = 'unknown'", "Imputed as monthly_charges x tenure_months", int(miss.sum())))

## 7. Missing `region`
13 rows (about 2%) have no region. Region is a nominal label, so the mode or a random value would invent information. These rows are labeled **`Unknown`** so they stay in the data and can be filtered out later.

In [ ]:
n = df["region"].isna().sum()
df["region"] = df["region"].fillna("Unknown")
log.append(("Missing values", "region is null", "Filled with 'Unknown' (no reliable way to infer)", int(n)))

## 8. Validate types and business rules

In [ ]:
df = df.astype({"age":"int64","tenure_months":"int64","support_tickets":"int64","late_payments":"int64",
                "monthly_charges_inr":"float64","total_charges_inr":"float64"})
for c in ["region","subscription_type","contract_type","payment_method","auto_pay","churn"]:
    df[c] = df[c].astype("category")


assert df.isna().sum().sum() == 0,              "nulls remain"
assert df.duplicated().sum() == 0,              "duplicates remain"
assert df["customer_id"].is_unique,             "customer_id not unique"
assert df["age"].between(18, 100).all(),        "age out of range"
assert df["tenure_months"].between(0, 120).all(), "tenure out of range"
assert (df["monthly_charges_inr"] > 0).all(),   "non-positive monthly charge"
assert (df["total_charges_inr"] >= 0).all(),    "negative total charge"
assert set(df["churn"].unique()) <= {"Yes","No"}
assert df.groupby("subscription_type", observed=True)["monthly_charges_inr"].max().max() < 2000, "outliers remain"
print("All validation checks passed")
df.info()

## 9. Before vs after

In [ ]:
summary = pd.DataFrame({
    "metric": ["rows","columns","missing cells","duplicate rows","non-numeric total_charges","max monthly charge"],
    "before": [len(df_raw), df_raw.shape[1], int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum()),
               int(pd.to_numeric(df_raw["TotalChargesINR"], errors="coerce").isna().sum()), df_raw["MonthlyChargesINR"].max()],
    "after":  [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()), 0, df["monthly_charges_inr"].max()],
})
summary

In [ ]:
pd.DataFrame(log, columns=["step","issue","action","affected"])

## 10. Export

In [ ]:
df.to_csv("customer_churn_clean.csv", index=False)
print("Saved customer_churn_clean.csv", df.shape)

## Summary of decisions
- **Duplicates:** 5 exact duplicates dropped (605 -> 600 rows).
- **`total_charges_inr`:** text `"unknown"` converted to a number and imputed from `monthly x tenure`, because the data shows this relationship holds closely.
- **`monthly_charges_inr`:** 7 values about 12x too large were corrected by dividing by 12, confirmed against total charges and plan pricing.
- **`region`:** 13 nulls labeled `Unknown`, because inferring a region would be guessing.
- **Standardization:** snake_case headers, trimmed Title Case categories, correct dtypes. There are no date fields in this dataset.